**Project Setup and Raw Data Ingetion Notebook**

## **In Databricks, we'll create:**
```text
Catalog
   ↓
retail_lakehouse (Catalog)
   ↓
raw (Schema)
   ↓
retail_files (Volume)
```

### Create Catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retail_lakehouse;

### CREATE SCHEMA

In [0]:
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.raw;

### CREATE VOLUME

In [0]:
CREATE VOLUME IF NOT EXISTS retail_lakehouse.raw.retail_files;

## Bronze schema

In [0]:
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.bronze;

### Create the Bronze Delta table Orders

In [0]:
CREATE TABLE IF NOT EXISTS retail_lakehouse.bronze.orders
(
    order_id BIGINT,
    customer_id BIGINT,
    order_date DATE,
    product_id STRING,
    quantity INT,
    unit_price DECIMAL(10,2),
    status STRING
)
USING DELTA;

**COPY INTO loads files from a file location into a Delta table** 

In [0]:
COPY INTO retail_lakehouse.bronze.orders
FROM (
  SELECT
    CAST(order_id AS BIGINT) AS order_id,
    CAST(customer_id AS BIGINT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    product_id,
    CAST(quantity AS INT) AS quantity,
    CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
    status
  FROM '/Volumes/retail_lakehouse/raw/retail_files/orders/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
);

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
13717,13717,0


**Validate record count**

In [0]:
SELECT COUNT(*) AS total_records
FROM retail_lakehouse.bronze.orders;

total_records
13717


**Preview data**

In [0]:
SELECT *
FROM retail_lakehouse.bronze.orders
LIMIT 20;

order_id,customer_id,order_date,product_id,quantity,unit_price,status
1,1071,2026-08-21,P016,2,22.26,COMPLETED
2,1140,2026-08-05,P006,2,341.58,RETURNED
3,1056,2026-08-14,P015,1,24.60,RETURNED
4,1184,2026-08-20,P042,1,285.01,RETURNED
5,1072,2026-08-14,P001,4,230.11,COMPLETED
6,1040,2026-08-23,P014,7,176.72,COMPLETED
7,1092,2026-08-04,P023,2,196.16,RETURNED
8,1138,2026-08-09,P008,1,367.57,CANCELLED
9,1159,2026-08-03,P024,5,416.41,RETURNED
10,1059,2026-08-07,P050,2,32.45,COMPLETED
